# Paper2Agent, in a weekend: turn a Python module into a callable agent tool-server

**Companion notebook for the Sep 25, 2026 post on Stanford's Paper2Agent** (Nature, Sep 16 2026).

Paper2Agent's headline result: 100 computational biology papers in, **74 converted to MCP agents**, 599 tools proposed, **593 passing automated validation** — and the agents beat "just give Claude the repo" (91.2% vs 82.7% on 300 questions).

The full system needs sandboxes, Docker, and an LLM to do the extraction. But the *shape* of the idea fits in one notebook. We'll play all six stages of the pipeline on a toy "paper" — a variant-effect scorer — using nothing but the Python standard library:

1. **Locate** — the "repo" is a small module we write to `/tmp`.
2. **Isolate** — we run it in a subprocess (our sandbox).
3. **Scan** — we read its docstrings (the "tutorials").
4. **Execute** — we run the functions and capture reference outputs.
5. **Extract** — we write tool specs (name, description, JSON schema).
6. **Assemble** — we serve them as an MCP-shaped JSON-RPC server over stdio.
7. **Validate** — outputs must match references within **3%**, exactly the paper's gate.

Then a client answers a question *through the agent* — and shows the provenance.

In [ ]:
# === STAGE 1-3: the "paper" and its repo ===
# A toy methods module, the way a paper's GitHub repo would hold it.
# We write it to /tmp/toy_paper.py so the *server* can import it too
# (in Paper2Agent, this is the "locate codebase" step).

TOY_PAPER = """
def gc_content(seq):
    \"\"\"Fraction of G/C bases in a DNA sequence.\"\"\"
    seq = seq.upper()
    return sum(1 for b in seq if b in "GC") / len(seq)

def score_variant(sequence, position, ref, alt):
    \"\"\"Predicted effect of a single-nucleotide variant, 0 (benign) to 1 (severe).
    Method (Fig. 2 of our toy paper): effect scales with how much the
    variant shifts local GC content in a 21bp window, plus a motif bonus
    if the variant sits in a CpG dinucleotide.
    \"\"\"
    sequence = sequence.upper()
    assert sequence[position] == ref.upper(), "ref base does not match sequence"
    lo, hi = max(0, position - 10), position + 11
    window = sequence[lo:hi]
    before = gc_content(window)
    mutated = sequence[:position] + alt.upper() + sequence[position + 1:]
    after = gc_content(mutated[lo:hi])
    score = min(1.0, abs(after - before) * 6.0)
    dinuc = (sequence[position - 1] + sequence[position + 1]) if 0 < position < len(sequence) - 1 else ""
    if "CG" in (sequence[position - 1:position + 1] if position > 0 else "") or \
       "CG" in (sequence[position:position + 2] if position < len(sequence) - 1 else ""):
        score = min(1.0, score + 0.25)  # CpG motif bonus
    return round(score, 4)

def rank_variants(sequence, variants):
    \"\"\"Rank a list of (position, ref, alt) tuples by predicted severity.\"\"\"
    scored = [(v, score_variant(sequence, *v)) for v in variants]
    return sorted(scored, key=lambda t: t[1], reverse=True)

def summarize_gene(name, scores):
    \"\"\"Aggregate per-variant scores into a gene-level burden summary.\"\"\"
    return {"gene": name, "n_variants": len(scores),
            "max_score": round(max(scores), 4), "mean_score": round(sum(scores) / len(scores), 4)}
"""

with open("/tmp/toy_paper.py", "w") as f:
    f.write(TOY_PAPER)

import sys
sys.path.insert(0, "/tmp")
import importlib, toy_paper
importlib.reload(toy_paper)

SEQ = "ATGCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATCGATC"
print("module loaded; functions:", [n for n in dir(toy_paper) if not n.startswith("_")])


## Stage 4: execute the tutorials, capture reference outputs

Paper2Agent runs the paper's own tutorials inside the sandbox and saves what came out — those become the ground truth every extracted tool must reproduce. We do the same:

In [ ]:
# === STAGE 4: execute "tutorials", capture reference outputs ===
# These are the paper's own worked examples. Their outputs are ground truth.
REF = {}  # (tool_name, args_tuple) -> expected output
REF[("score_variant", (SEQ, 10, "T", "G"))] = toy_paper.score_variant(SEQ, 10, "T", "G")
REF[("score_variant", (SEQ, 11, "C", "A"))] = toy_paper.score_variant(SEQ, 11, "C", "A")
REF[("rank_variants", (SEQ, ((10, "T", "G"), (11, "C", "A"), (20, "G", "T"))))] = \
    toy_paper.rank_variants(SEQ, [(10, "T", "G"), (11, "C", "A"), (20, "G", "T")])
REF[("summarize_gene", ("GPR137", (0.1, 0.5, 0.9)))] = toy_paper.summarize_gene("GPR137", [0.1, 0.5, 0.9])

for (tool, args), out in REF.items():
    print(f"{tool}{args[1:] if len(args) > 1 else ''}  ->  {out}")
print(f"\n{len(REF)} reference outputs captured. In Paper2Agent this is fully automated;")
print("numerical outputs must later match within 3%, figures by perceptual hash.")


## Stage 5: extract parameterized tool specs

The agent reads the code + docstrings and proposes *tools*: a name, a human description, and a JSON input schema. This is the artifact an MCP server needs — nothing more.

In [ ]:
# === STAGE 5: extract MCP tool specs ===
TOOL_SPECS = [
    {"name": "score_variant",
     "description": "Score a single-nucleotide variant's predicted effect, 0 (benign) to 1 (severe).",
     "input_schema": {"type": "object",
                      "properties": {"sequence": {"type": "string"},
                                     "position": {"type": "integer"},
                                     "ref": {"type": "string"}, "alt": {"type": "string"}},
                      "required": ["sequence", "position", "ref", "alt"]}},
    {"name": "rank_variants",
     "description": "Rank (position, ref, alt) variant tuples by predicted severity, worst first.",
     "input_schema": {"type": "object",
                      "properties": {"sequence": {"type": "string"},
                                     "variants": {"type": "array"}},
                      "required": ["sequence", "variants"]}},
    {"name": "summarize_gene",
     "description": "Aggregate per-variant scores into a gene-level burden summary.",
     "input_schema": {"type": "object",
                      "properties": {"name": {"type": "string"},
                                     "scores": {"type": "array"}},
                      "required": ["name", "scores"]}},
]
print(f"extracted {len(TOOL_SPECS)} tool specs:")
for t in TOOL_SPECS:
    print(f"  - {t['name']}: {t['description']}")


## Stage 6: assemble the MCP server

Real MCP servers speak JSON-RPC 2.0 over stdio with three methods: `initialize`, `tools/list`, `tools/call`. Ours does exactly that — ~40 lines, stdlib only. The tools are **locked**: the server can only run the extracted functions, never improvise (the paper's anti-hallucination measure).

In [ ]:
# === STAGE 6: assemble the MCP server (JSON-RPC over stdio, stdlib only) ===
import json  # needed here to embed the tool specs into the server script
SERVER_CODE = """
import sys, json
sys.path.insert(0, "/tmp")
import toy_paper

SPECS = __SPECS__

def call_tool(name, args):
    fn = {"score_variant": toy_paper.score_variant,
          "rank_variants": toy_paper.rank_variants,
          "summarize_gene": toy_paper.summarize_gene}[name]
    if name == "rank_variants":
        args = dict(args); args["variants"] = [tuple(v) for v in args["variants"]]
    return fn(**args)

def handle(req):
    m = req.get("method")
    if m == "initialize":
        return {"protocolVersion": "2024-11-05",
                "serverInfo": {"name": "toy-paper-agent", "version": "1.0.0"}}
    if m == "tools/list":
        return {"tools": SPECS}
    if m == "tools/call":
        p = req["params"]
        return {"content": [{"type": "text",
                             "text": json.dumps(call_tool(p["name"], p.get("arguments", {})))}]}
    return {"error": "unknown method " + str(m)}

for line in sys.stdin:
    line = line.strip()
    if not line:
        continue
    req = json.loads(line)
    resp = {"jsonrpc": "2.0", "id": req.get("id")}
    try:
        resp["result"] = handle(req)
    except Exception as e:
        resp["error"] = {"code": -32000, "message": str(e)}
    print(json.dumps(resp), flush=True)
"""

with open("/tmp/paper2agent_server.py", "w") as f:
    f.write(SERVER_CODE.replace("__SPECS__", json.dumps(TOOL_SPECS)))
print("server written to /tmp/paper2agent_server.py")


## The client: what any chat agent does with an MCP server

A chat agent never sees the repo. It spawns the server, asks what tools exist, and calls them. That's the whole interaction pattern Paper2Agent's 593 validated tools expose.

In [ ]:
# === CLIENT: spawn the server, speak JSON-RPC ===
import subprocess, json

class MCPClient:
    def __init__(self):
        self.proc = subprocess.Popen(
            [sys.executable, "/tmp/paper2agent_server.py"],
            stdin=subprocess.PIPE, stdout=subprocess.PIPE,
            text=True, bufsize=1)
        self._id = 0

    def call(self, method, params=None):
        self._id += 1
        req = {"jsonrpc": "2.0", "id": self._id, "method": method}
        if params is not None:
            req["params"] = params
        self.proc.stdin.write(json.dumps(req) + "\n")
        self.proc.stdin.flush()
        resp = json.loads(self.proc.stdout.readline())
        if "error" in resp:
            raise RuntimeError(resp["error"])
        return resp["result"]

    def close(self):
        self.proc.terminate()

client = MCPClient()
print("initialize ->", client.call("initialize")["serverInfo"])
tools = client.call("tools/list")["tools"]
print(f"tools/list -> {len(tools)} tools:", [t["name"] for t in tools])


## The validation gate: reproduce the paper or don't ship

This is the step that makes Paper2Agent more than a wrapper generator. Every tool is called through the server and its output compared against the reference outputs from Stage 4. Paper2Agent's bar: **numeric outputs within 3%**, figures by perceptual hash. A tool that can't reproduce the paper's own outputs never ships — 6 of 599 proposed tools failed exactly here.

In [ ]:
# === VALIDATION GATE: 3% tolerance, like the paper ===
def close_enough(a, b, tol=0.03):
    if isinstance(a, float) or isinstance(b, float):
        return abs(a - b) <= tol * max(1e-9, abs(b))
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(close_enough(a[k], b[k], tol) for k in a)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(close_enough(x, y, tol) for x, y in zip(a, b))
    return a == b

locked = []
for (tool, args), expected in REF.items():
    params = dict(zip(TOOL_SPECS[[t["name"] for t in TOOL_SPECS].index(tool)]
                                 ["input_schema"]["required"], args))
    got = json.loads(client.call("tools/call",
                                 {"name": tool, "arguments": params})
                     ["content"][0]["text"])
    ok = close_enough(got, expected)
    print(("PASS " if ok else "FAIL ") + f"{tool}  server={got}  ref={expected}")
    assert ok, f"tool {tool} failed validation"
    locked.append(tool)

print(f"\n{len(set(locked))}/{len(TOOL_SPECS)} tools locked against hallucination. Agent ready.")


## Ask the agent a question

No repo access, no code reading — just natural-language intent routed through the verified tools. This mirrors the paper's 300-question benchmark setup.

In [ ]:
# === ANSWER A QUESTION THROUGH THE AGENT ===
# "Among these candidate variants in GPR137, which is most damaging, and what's the gene burden?"
candidates = [(10, "T", "G"), (11, "C", "A"), (20, "G", "T"), (30, "T", "A"), (41, "A", "T")]

ranked = json.loads(client.call(
    "tools/call",
    {"name": "rank_variants",
     "arguments": {"sequence": SEQ, "variants": candidates}})["content"][0]["text"])
ranked = [(tuple(v), s) for v, s in ranked]
burden = json.loads(client.call(
    "tools/call",
    {"name": "summarize_gene",
     "arguments": {"name": "GPR137", "scores": [s for _, s in ranked]}})["content"][0]["text"])

worst, worst_score = ranked[0]
print(f"Q: which of {len(candidates)} candidate variants in GPR137 is most damaging?")
print(f"A: position {worst[0]} ({worst[1]}->{worst[2]}), score {worst_score:.4f}")
print(f"   gene burden: max={burden['max_score']}, mean={burden['mean_score']} over {burden['n_variants']} variants")
print("\nProvenance: every number above came from a locked tool that reproduced")
print("the paper's own reference outputs within 3% — not from the LLM's memory.")
client.close()


## Takeaways

- **The paper becomes the API.** Paper2Agent's bet: scientific knowledge should ship as *runnable tools*, not PDFs. Our 40-line server is the same shape — `tools/list` + `tools/call` — as the 593 validated tools Stanford produced.
- **Validation is the product.** Anyone can wrap code in JSON-RPC. The 3%-tolerance reproduce-the-paper gate (plus figure perceptual-hash checks, plus locking tools against hallucination) is what makes the agents trustworthy — and it's why 26 of 100 papers *failed* conversion: missing code, missing data, unbuildable environments.
- **Structure beats access.** On 300 questions, the agents hit **91.2%** vs **82.7%** for Claude handed the same repos directly. The model is the same; the *packaging* of knowledge is what changed — parameterized, validated, callable tools instead of a pile of files.
- **Agents can team up.** Three paper-agents (AlphaGenome + MPRA/scCRISPRi + Perturb-seq) jointly prioritized **GPR137** at the psoriasis locus rs887314 — a link none of the three papers made alone. Papers as agents means papers can *collaborate*.
- **The honest limits.** Agentification is a reproducibility *certificate*, not a discovery engine: open-ended hypothesis selection stays human-in-the-loop, and every MCP server rots as its upstream dependencies drift.

*Try it:* swap in your own module for `toy_paper` — any deterministic functions with docstrings — and run this notebook end to end. You've just done stages 1–6 of Paper2Agent by hand.